# Greek EPU Index — walkthrough

This notebook walks through the published monthly index series and runs the diagnostics that do not require the raw newspaper corpus. To reproduce the full pipeline (article scoring, monthly aggregation, merge with the bond spread, complete econometric battery) use the CLI:

```bash
python -m src.pipeline --articles ... --spread ... --output data/processed
```

In [ ]:
import sys
from pathlib import Path

# Allow imports from src/ when notebook is run from notebooks/
ROOT = Path.cwd().parent if (Path.cwd() / 'notebooks').exists() is False else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.econometrics import stationarity_battery, block_bootstrap_slope

## Load the headline series

In [ ]:
epu = pd.read_csv(ROOT / 'data' / 'epu_timeseries.csv')
epu['year_month'] = pd.to_datetime(epu['year_month'])
epu = epu.sort_values('year_month').reset_index(drop=True)
print(f'Observations: {len(epu)}')
print(f'Range: {epu.year_month.min().strftime("%Y-%m")} to {epu.year_month.max().strftime("%Y-%m")}')
epu.head()

## Plot the index

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(epu['year_month'], epu['epu_index_detrended'], linewidth=1.2)
ax.axhline(100, color='gray', linewidth=0.5)
for ds, label in [('2010-05', 'First Bailout'), ('2012-03', 'PSI'), ('2015-07', 'Referendum'),
                  ('2018-08', 'End of Bailouts'), ('2020-03', 'COVID-19'), ('2022-02', 'Ukraine War')]:
    ax.axvline(pd.Timestamp(ds), color='gray', linestyle='--', alpha=0.3)
ax.set_title('Greek EPU index (detrended), 2010-2026')
ax.set_ylabel('Index')
plt.tight_layout()
plt.show()

## Stationarity battery

In [ ]:
for series_name, series in [
    ('EPU levels', epu['epu_index_detrended']),
    ('EPU first differences', epu['epu_index_detrended'].diff()),
]:
    r = stationarity_battery(series, series_name)
    print(f'{series_name}:')
    print(f'  ADF stat={r.adf_stat:.3f}, p={r.adf_pvalue:.4f}')
    print(f'  KPSS stat={r.kpss_stat:.3f}, p={r.kpss_pvalue:.4f}')
    print(f'  Verdict: {r.verdict}')
    print()

## Subsample summary

The index has substantially different mean and variance across the two halves of the sample, reflecting the structural shift from the Greek debt crisis to the post-bailout era.

In [ ]:
mid = len(epu) // 2
h1, h2 = epu.iloc[:mid], epu.iloc[mid:]
summary = pd.DataFrame({
    'period': [f"{h1.year_month.min():%Y-%m} – {h1.year_month.max():%Y-%m}",
               f"{h2.year_month.min():%Y-%m} – {h2.year_month.max():%Y-%m}"],
    'n': [len(h1), len(h2)],
    'mean': [h1['epu_index_detrended'].mean(), h2['epu_index_detrended'].mean()],
    'std': [h1['epu_index_detrended'].std(), h2['epu_index_detrended'].std()],
}, index=['First half', 'Second half'])
summary

## Next steps

To compare the index against the bond spread, run the full pipeline with both data sources (see `docs/data_schema.md` for input formats) and inspect `data/processed/diagnostics.json`.